# Big Data Architecture Patterns

Big Data architecture defines how data is **collected, stored, processed, and served** at scale. Several patterns have emerged over time, each with different trade-offs between **latency**, **complexity**, and **cost**.

## Overview

| Pattern | Approach | Best For | Complexity |
|---------|----------|----------|------------|
| **Lambda Architecture** | Batch layer + Speed layer + Serving layer | Mixed batch and real-time | High |
| **Kappa Architecture** | Single streaming pipeline for all data | Primarily streaming workloads | Medium |
| **Data Lakehouse** | Unified batch + streaming on open table formats | Modern analytics + ML | Medium |
| **Data Mesh** | Domain-owned data products, federated governance | Large enterprises | High |
| **Medallion (Delta)** | Bronze → Silver → Gold layers | Databricks / Delta Lake | Low–Medium |

## Lambda Architecture

Introduced by **Nathan Marz** (~2011). Designed to handle both **batch** and **real-time** data processing by running two separate processing layers in parallel.

```
                     ┌─────────────────────┐
                     │    Source Data       │
                     └──────────┬──────────┘
                                │
               ┌────────────────┴─────────────────┐
               ▼                                  ▼
    ┌─────────────────────┐            ┌──────────────────────┐
    │    Batch Layer       │            │    Speed Layer        │
    │  (High latency,      │            │  (Low latency,        │
    │   high accuracy)     │            │   approximate)        │
    │  e.g., Spark Batch   │            │  e.g., Structured     │
    │                      │            │  Streaming / Kafka    │
    └──────────┬──────────┘            └──────────┬───────────┘
               │                                  │
               └────────────────┬─────────────────┘
                                ▼
                     ┌─────────────────────┐
                     │    Serving Layer     │
                     │  Merges batch +      │
                     │  real-time views     │
                     └──────────┬──────────┘
                                ▼
                           Analytics / BI
```

### Three Layers

| Layer | Role | Characteristics |
|-------|------|-----------------|
| **Batch Layer** | Recomputes views over the complete historical dataset on a schedule | High accuracy, high latency, fault-tolerant |
| **Speed Layer** | Processes only the most recent data incrementally to fill the latency gap | Low latency, approximate, temporary |
| **Serving Layer** | Merges results from batch and speed layers to answer queries | Responds to ad-hoc queries |

### Pros & Cons
| Pros | Cons |
|------|------|
| Fault-tolerant (recompute from raw data) | Must maintain **two separate codebases** |
| Handles both batch and real-time | High operational complexity |
| Query flexibility | Results can be inconsistent between layers |

## Kappa Architecture

Proposed by **Jay Kreps** (LinkedIn, 2014) as a **simplification of Lambda**. The core idea: treat everything as a stream, use **a single processing layer** for both real-time and historical data.

```
  Source Data
      │
      ▼
  Message Queue (e.g., Kafka – retains all history)
      │
      ▼
  Single Stream Processing Layer
  (e.g., Spark Structured Streaming)
      │
      ▼
   Serving Layer / Output Store
      │
      ▼
  Analytics / BI
```

- To **reprocess** historical data: replay the stream from the beginning of the message queue log.
- Eliminates the need for a separate batch layer.

### Pros & Cons
| Pros | Cons |
|------|------|
| Simpler — one codebase | Requires a **log-based message broker** with long retention (Kafka) |
| Easier to maintain | Not ideal when batch recomputes are complex or very large |
| Lower operational overhead | Historical replays can be expensive |

## Lambda vs. Kappa — Quick Comparison

| Factor | Lambda | Kappa |
|--------|--------|-------|
| **Processing layers** | 2 (batch + speed) | 1 (streaming only) |
| **Code complexity** | High (duplicate logic) | Low |
| **Reprocessing** | Re-run batch job | Replay Kafka topic |
| **Latency** | Low (via speed layer) | Low |
| **Accuracy** | High (via batch layer) | Depends on state management |
| **Best for** | Complex batch + real-time | Mostly streaming workloads |
| **In Databricks** | Batch jobs + Structured Streaming | Structured Streaming with Delta |

## Data Lakehouse Architecture ⭐

The **Data Lakehouse** combines the best of **Data Lakes** (flexibility, low cost) and **Data Warehouses** (ACID transactions, performance, schema enforcement). This is the **primary architecture in Databricks**.

### Data Lake vs. Data Warehouse vs. Data Lakehouse

| Feature | Data Lake | Data Warehouse | Data Lakehouse |
|---------|-----------|----------------|----------------|
| **Schema** | Schema-on-read | Schema-on-write | Both |
| **Data types** | All (structured, semi, unstructured) | Structured only | All |
| **ACID transactions** | ❌ | ✅ | ✅ (via Delta Lake) |
| **Performance** | Low (no indexing) | High | High (file skipping, caching) |
| **Cost** | Low | High | Low–Medium |
| **ML support** | Good | Poor | Excellent |
| **Concurrency** | Limited | High | High |
| **Example** | S3 + Hive | Snowflake, Redshift | **Databricks + Delta Lake** |

### Key enabling technology: **Delta Lake**
- Open-source storage layer that brings ACID transactions to data lakes.
- Stores data as **Parquet files** + a **transaction log** (`_delta_log/`).
- Supports **time travel** (query older versions of data).
- Enables **schema evolution** and **schema enforcement**.
- Powers **Structured Streaming** with exactly-once semantics.

## Medallion Architecture (Databricks Standard) ⭐

A **data design pattern** used within the Lakehouse that organizes data into three quality layers.

```
  Raw Sources
      │
      ▼
  ┌─────────┐     ┌──────────┐     ┌──────────┐
  │ BRONZE  │ ──▶ │  SILVER  │ ──▶ │   GOLD   │
  │  (Raw)  │     │(Cleaned) │     │(Curated) │
  └─────────┘     └──────────┘     └──────────┘
                                        │
                        ┌───────────────┼───────────────┐
                        ▼               ▼               ▼
                    Analytics          ML           Operational
```

### Layer Details

#### 🥉 Bronze Layer
- **What**: Raw, unprocessed data ingested from source systems.
- **Why**: Preserves original data for auditability and reprocessing.
- **State**: May have duplicates, nulls, schema inconsistencies.
- **Format**: Often Delta, Parquet, or JSON as-is.
- **Retention**: Long-term (source of truth for replay).

#### 🥈 Silver Layer
- **What**: Cleaned, validated, and joined data.
- **Why**: Provides a reliable, consistent dataset for analytics and feature engineering.
- **Operations**: Deduplication, null handling, schema normalization, PII masking, joining reference tables.
- **Format**: Delta tables with enforced schema.

#### 🥇 Gold Layer
- **What**: Aggregated, business-logic-applied data ready for consumption.
- **Why**: Directly supports BI dashboards, reports, and ML models.
- **Operations**: Aggregations, KPI calculations, feature computation.
- **Example**: Daily revenue by region, customer lifetime value.

### Best Practices
- Use **Delta Live Tables (DLT)** to declare and automate Medallion pipelines.
- Define **data quality expectations** at the Silver layer (quarantine bad records).
- Gold tables should be **denormalized** (optimized for query speed, not storage).

## Data Mesh (Brief Overview)

A **sociotechnical architecture** proposed by **Zhamak Dehghani** (2019). Shifts from centralized data ownership to **domain-oriented, decentralized data ownership**.

### Four Principles
1. **Domain-oriented ownership**: Each business domain owns and publishes its own data as a product.
2. **Data as a product**: Data is treated like a product — discoverable, reliable, self-describing.
3. **Self-serve data platform**: A platform (e.g., Databricks + Unity Catalog) that allows domains to build independently.
4. **Federated computational governance**: Shared policies enforced centrally, implemented locally (e.g., Unity Catalog policies).

### Databricks & Data Mesh
- **Unity Catalog** enables federated governance across domains.
- **Delta Sharing** allows secure cross-domain / cross-cloud data sharing.
- Each domain can have its own workspace but share assets via a central metastore.

## MapReduce — The Foundation of Big Data Processing

**MapReduce** (Google, 2004 — Hadoop 2006) was the first widely-adopted framework for distributed batch processing of large datasets. While largely superseded by Spark, understanding it is fundamental.

### How it Works

```
Input Data (split across nodes)
      │
      ▼
  MAP Phase
  (Each node processes its local data split independently)
  Input: (key, value) pairs
  Output: intermediate (key, value) pairs
      │
      ▼
  SHUFFLE & SORT
  (Group all values for the same key together, across nodes)
      │
      ▼
  REDUCE Phase
  (Aggregates grouped values into final output)
  Input: (key, [list of values])
  Output: final (key, value) pairs
```

### Classic Example: Word Count
| Phase | Input | Output |
|-------|-------|--------|
| Map | `"hello world hello"` | `(hello,1), (world,1), (hello,1)` |
| Shuffle | Group by key | `(hello,[1,1]), (world,[1])` |
| Reduce | `(hello,[1,1])` | `(hello, 2)` |

### MapReduce vs. Apache Spark

| Aspect | MapReduce | Apache Spark |
|--------|-----------|---------------|
| **Storage** | Writes to disk after every step | In-memory (RDDs, caching) |
| **Speed** | Slow (disk I/O between stages) | Up to **100×** faster in memory |
| **API** | Low-level Java | High-level (Python, Scala, SQL, R) |
| **Streaming** | Not supported natively | Structured Streaming built-in |
| **ML** | Mahout (limited) | MLlib + integration with MLflow |
| **Use today** | Legacy systems | Modern default |